# 04 - Klasyfikacja: czy kierowca skończy na podium?

**Czy na podstawie tego, co wiemy przed startem, da się przewidzieć podium?**

Korzystamy z cech **przedwyścigowych**: pozycji startowej, zespołu oraz różnicy czasu do najszybszego okrążenia kwalifikacji. Podobnie jak w notebooku 03, prowadzimy to jak realny proces - z dwiema pułapkami, w które celowo wchodzimy, żeby pokazać, dlaczego są groźne:

1. pułapka **metryki** - dlaczego sama trafność (`accuracy`) tutaj myli,
2. pułapka **wycieku danych** - dlaczego nie wolno użyć cech z przebiegu wyścigu.

Następnie przechodzimy do uczciwej oceny (`ROC-AUC`, `PR-AUC`) i porównania algorytmów wraz z czasem ich działania.

In [1]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

dr = pd.read_parquet('../data/driver_race.parquet')
dr = dr.dropna(subset=['GridPosition']).reset_index(drop=True)
print('wierszy:', len(dr), '| podiów:', int(dr.podium.sum()), f'({dr.podium.mean():.1%})')
print('sezony:', sorted(dr.Season.unique()))

wierszy: 918 | podiów: 138 (15.0%)
sezony: [np.int64(2023), np.int64(2024)]


## Cechy i sposób walidacji

Bierzemy wyłącznie cechy **przedwyścigowe**. Cechę `quali_gap_s` dołączamy tylko wtedy, gdy jest realnie wypełniona - w pierwotnej wersji projektu była w całości pusta (opisaliśmy to w notebooku 01, Krok 1).

Danych jest niewiele, a klasy są niezbalansowane, dlatego zamiast jednego podziału stosujemy **walidację krzyżową z warstwowaniem** (`StratifiedKFold`) i oceniamy model na predykcjach *out-of-fold* (dla każdego wiersza predykcja pochodzi z modelu, który go nie widział na treningu).

In [2]:
cat_cols = ['Team']
num_cols = ['GridPosition']
if dr['quali_gap_s'].notna().sum() > 0.5 * len(dr):
    num_cols.append('quali_gap_s')
    dr['quali_gap_s'] = dr['quali_gap_s'].fillna(dr['quali_gap_s'].median())
    print('quali_gap_s UŻYTE jako cecha (wypełnione:', dr.quali_gap_s.notna().sum(), ')')
else:
    print('quali_gap_s pominięte — za dużo braków')

features = cat_cols + num_cols
X = dr[features].copy()
y = dr['podium'].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('cechy:', features)

quali_gap_s UŻYTE jako cecha (wypełnione: 918 )
cechy: ['Team', 'GridPosition', 'quali_gap_s']


## Pułapka pierwsza - dlaczego sama trafność myli

Wyobraźmy sobie najprostszy możliwy "model": **zawsze twierdzi, że podium nie będzie**. Zobaczmy, jaką osiągnie trafność:

In [3]:
zawsze_nie = np.zeros_like(y)
print(f'„zawsze NIE podium" — accuracy = {accuracy_score(y, zawsze_nie):.1%}')
print('...model, który nie przewiduje NICZEGO, ma ~85% trafności.')
print('Dlatego accuracy jest tu bezużyteczne — liczą się precision/recall, ROC-AUC, PR-AUC.')

„zawsze NIE podium" — accuracy = 85.0%
...model, który nie przewiduje NICZEGO, ma ~85% trafności.
Dlatego accuracy jest tu bezużyteczne — liczą się precision/recall, ROC-AUC, PR-AUC.


## Pułapka druga - kuszący wyciek danych

Gdybyśmy chcieli sztucznie podbić wynik, moglibyśmy dorzucić cechę z **przebiegu** wyścigu - na przykład liczbę zdobytych punktów. Zobaczmy, co się wtedy stanie i dlaczego jest to oszustwo:

In [4]:
# CELOWY BLAD: dorzucamy 'Points' (znane DOPIERO po wyscigu)
X_leak = dr[features + ['Points']].copy()
pre_leak = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols + ['Points'])])
leak_model = Pipeline([('pre', pre_leak),
                       ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
proba_leak = cross_val_predict(leak_model, X_leak, y, cv=cv, method='predict_proba')[:, 1]
print(f'Z wyciekiem (Points): ROC-AUC = {roc_auc_score(y, proba_leak):.3f}  <- podejrzanie idealne')

Z wyciekiem (Points): ROC-AUC = 1.000  <- podejrzanie idealne


Wynik wygląda idealnie - `ROC-AUC` bliskie `1.0`. I właśnie dlatego powinien wzbudzić podejrzliwość.

Liczba punktów (`Points`) jest przyznawana **za** wynik wyścigu - kto zdobył ich dużo, ten z definicji był wysoko. Model nie tyle przewiduje podium, co odczytuje je z niemal gotowej odpowiedzi. To klasyczny **wyciek danych**: cecha, której w momencie przewidywania (przed startem) po prostu nie znamy.

Dlatego pozostajemy wyłącznie przy cechach przedwyścigowych.

## Model główny - regresja logistyczna

Wracamy do uczciwych cech. Parametr `class_weight='balanced'` kompensuje przewagę liczebną klasy "nie-podium", a ocenę prowadzimy na predykcjach *out-of-fold*.

In [5]:
pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)])
logit = Pipeline([('pre', pre),
                  ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
proba_logit = cross_val_predict(logit, X, y, cv=cv, method='predict_proba')[:, 1]
pred_logit = (proba_logit >= 0.5).astype(int)
print('Regresja logistyczna (cechy przedwyścigowe):')
print(f'  ROC-AUC = {roc_auc_score(y, proba_logit):.3f}  PR-AUC = {average_precision_score(y, proba_logit):.3f}')
print(classification_report(y, pred_logit, target_names=['nie podium', 'podium']))

Regresja logistyczna (cechy przedwyścigowe):
  ROC-AUC = 0.921  PR-AUC = 0.703
              precision    recall  f1-score   support

  nie podium       0.98      0.80      0.88       780
      podium       0.44      0.89      0.59       138

    accuracy                           0.81       918
   macro avg       0.71      0.85      0.73       918
weighted avg       0.90      0.81      0.84       918



## Jak rozumieć ROC-AUC i PR-AUC?

Skoro to na tych metrykach oprzemy ocenę, warto wiedzieć, co dokładnie mierzą. Najprostsza interpretacja `ROC-AUC` jest taka: bierzemy losowo jednego kierowcę, który **był** na podium, i jednego, który **nie był**. `ROC-AUC` to prawdopodobieństwo, że model przypisał wyższą szansę temu właściwemu. Sprawdźmy to wprost na naszych predykcjach:

In [6]:
pod = proba_logit[y == 1]      # prawdopodobieństwa dla prawdziwych podiów
nie = proba_logit[y == 0]      # prawdopodobieństwa dla nie-podiów
roznice = pod[:, None] - nie[None, :]          # każda para (podium, nie-podium)
auc_z_par = (roznice > 0).mean() + 0.5 * (roznice == 0).mean()
print(f'par (podium x nie-podium): {roznice.size}')
print(f'model dał wyższą szansę podium w: {auc_z_par:.1%} par')
print(f'ROC-AUC ze sklearn:           {roc_auc_score(y, proba_logit):.3f}  (to samo)')

par (podium x nie-podium): 107640
model dał wyższą szansę podium w: 92.1% par
ROC-AUC ze sklearn:           0.921  (to samo)


Czyli `ROC-AUC = 0.5` oznaczałoby rzut monetą (model nie odróżnia podium od reszty), a `1.0` model idealny. Nasze `~0.92` znaczy, że w ponad dziewięciu na dziesięć przypadków model poprawnie wskazuje, który z dwóch kierowców miał większą szansę na podium.

Dlaczego podajemy też `PR-AUC` (pole pod krzywą precyzja-czułość)? Bo `ROC-AUC` bywa zbyt optymistyczne przy rzadkiej klasie. Podia to tylko `~15%` przypadków, a `PR-AUC` skupia się właśnie na tym, jak dobrze model radzi sobie z **wykrywaniem tej rzadkiej klasy** - jest więc surowszą i bardziej wymagającą miarą. Punktem odniesienia jest dla niej sam odsetek podiów (`~0.15`), więc nasze `~0.70` to wynik wyraźnie ponad przypadek.

## Benchmark - porównanie klasyfikatorów

Porównujemy kilka algorytmów na tych samych danych i tej samej walidacji. Notujemy `ROC-AUC`, `PR-AUC` oraz **czas pełnej walidacji krzyżowej**. `DummyClassifier`, przewidujący zawsze klasę większościową, jest punktem odniesienia.

In [7]:
def make(est, scale=True):
    steps = [('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)]
    steps.append(('num', StandardScaler() if scale else 'passthrough', num_cols))
    return Pipeline([('pre', ColumnTransformer(steps)), ('clf', est)])

models = {
    'Dummy (większość)': make(DummyClassifier(strategy='most_frequent')),
    'Logistic Regression': make(LogisticRegression(class_weight='balanced', max_iter=1000)),
    'kNN (k=15)': make(KNeighborsClassifier(n_neighbors=15)),
    'Random Forest': make(RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                 n_jobs=-1, random_state=42), scale=False),
    'HistGradientBoosting': make(HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=200, l2_regularization=1.0,
        class_weight='balanced', random_state=42), scale=False),
}

rows = []
for name, model in models.items():
    t = time.perf_counter()
    try:
        proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
        auc = roc_auc_score(y, proba); ap = average_precision_score(y, proba)
    except Exception:
        pred = cross_val_predict(model, X, y, cv=cv)
        auc = roc_auc_score(y, pred); ap = average_precision_score(y, pred)
    dt = time.perf_counter() - t
    rows.append(dict(model=name, ROC_AUC=auc, PR_AUC=ap, cv_time_s=dt))
    print(f'{name:<22} ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}  czas_CV={dt:.2f}s')

bench = pd.DataFrame(rows).set_index('model').round(3)
bench

Dummy (większość)      ROC-AUC=0.500  PR-AUC=0.150  czas_CV=0.03s


Logistic Regression    ROC-AUC=0.921  PR-AUC=0.703  czas_CV=0.05s
kNN (k=15)             ROC-AUC=0.918  PR-AUC=0.666  czas_CV=0.04s


Random Forest          ROC-AUC=0.907  PR-AUC=0.607  czas_CV=1.74s


HistGradientBoosting   ROC-AUC=0.912  PR-AUC=0.643  czas_CV=0.84s


,ROC_AUC,PR_AUC,cv_time_s
model,,,
Dummy (większość),0.500,0.150,0.032
Logistic Regression,0.921,0.703,0.050
kNN (k=15),0.918,0.666,0.044
Random Forest,0.907,0.607,1.742
HistGradientBoosting,0.912,0.643,0.840


In [8]:
b = bench.reset_index()
fig = px.scatter(b, x='cv_time_s', y='ROC_AUC', text='model', color='model',
                 title='ROC-AUC (wyżej=lepiej) vs czas walidacji',
                 labels={'cv_time_s': 'czas 5-fold CV [s]', 'ROC_AUC': 'ROC-AUC'})
fig.update_traces(textposition='top center', marker_size=12)
fig.update_layout(showlegend=False)
fig.show()

Wynik jest pouczający: przy małym zbiorze (dwa sezony, raptem `138` podiów) prosta **regresja logistyczna** wypada równie dobrze lub lepiej niż modele drzewiaste - a przy tym jest szybsza i łatwiejsza do interpretacji. To typowe: bardziej złożony model potrzebuje znacznie więcej danych, by jego przewaga się ujawniła.

## Diagnostyka modelu głównego

Przyglądamy się trzem rzeczom: macierzy pomyłek, krzywym `ROC` oraz `Precision-Recall`, a na końcu współczynnikom modelu - by zobaczyć, które cechy najmocniej wpływają na szansę podium.

In [9]:
cm = confusion_matrix(y, pred_logit)
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                x=['pred: nie', 'pred: podium'],
                y=['rzecz.: nie', 'rzecz.: podium'],
                title='Macierz pomyłek — regresja logistyczna')
fig.show()

In [10]:
fpr, tpr, _ = roc_curve(y, proba_logit)
prec, rec, _ = precision_recall_curve(y, proba_logit)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, name='ROC (logit)', mode='lines'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(dash='dash'), name='losowy'))
fig.update_layout(title='Krzywa ROC', xaxis_title='FPR (1 - swoistość)', yaxis_title='TPR (czułość)')
fig.show()

fig2 = px.area(x=rec, y=prec, title='Krzywa Precision-Recall',
               labels={'x': 'Recall (czułość)', 'y': 'Precision'})
fig2.add_hline(y=y.mean(), line_dash='dash', annotation_text='baza (odsetek podiów)')
fig2.show()

In [11]:
logit.fit(X, y)
ohe = logit.named_steps['pre'].named_transformers_['cat']
names = list(ohe.get_feature_names_out(cat_cols)) + num_cols
coefs = logit.named_steps['clf'].coef_[0]
imp = pd.DataFrame({'cecha': names, 'współczynnik': coefs}).sort_values('współczynnik')
fig = px.bar(imp, x='współczynnik', y='cecha', orientation='h',
             title='Współczynniki regresji logistycznej (log-szanse)')
fig.show()
print('Dodatni współczynnik = zwiększa szansę na podium. Ujemny GridPosition = im dalej z tyłu, tym gorzej.')

Dodatni współczynnik = zwiększa szansę na podium. Ujemny GridPosition = im dalej z tyłu, tym gorzej.


## Czy więcej sezonów pomoże?

To samo pytanie zadaliśmy przy regresji - i warto je tu powtórzyć, bo sytuacja jest inna. Klasyfikacja ma do dyspozycji znacznie mniej danych: nie dziesiątki tysięcy okrążeń, lecz kilkaset wyścigów, z czego podia to tylko `~15%`. Tutaj więcej sezonów oznacza **więcej podiów** - a tego właśnie modelowi brakuje. Sprawdźmy, czy to pomaga.

Korzystamy z szerszego zbioru `driver_race_all` (wszystkie pobrane sezony) i liczymy jakość dla rosnącej liczby sezonów, dokładanych od najnowszego wstecz:

In [12]:
dr_all = pd.read_parquet('../data/driver_race_all.parquet').dropna(subset=['GridPosition'])
seasons_all = sorted(dr_all['Season'].unique())
print('dostępne sezony:', seasons_all, '| wierszy:', len(dr_all),
      '| podiów:', int(dr_all.podium.sum()))

def ocena_clf(df, use_team=True):
    df = df.copy()
    cat = ['Team'] if use_team else []
    num = ['GridPosition']
    if df['quali_gap_s'].notna().sum() > 0.5 * len(df):
        num.append('quali_gap_s')
        df['quali_gap_s'] = df['quali_gap_s'].fillna(df['quali_gap_s'].median())
    Xx, yy = df[cat + num], df['podium'].values
    steps = ([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)]
             if cat else [])
    steps.append(('num', StandardScaler(), num))
    m = Pipeline([('pre', ColumnTransformer(steps)),
                  ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
    cv5 = StratifiedKFold(5, shuffle=True, random_state=42)
    pr = cross_val_predict(m, Xx, yy, cv=cv5, method='predict_proba')[:, 1]
    return roc_auc_score(yy, pr), average_precision_score(yy, pr)

rows = []
for k in range(1, len(seasons_all) + 1):
    chosen = seasons_all[-k:]
    sub = dr_all[dr_all['Season'].isin(chosen)]
    auc, ap = ocena_clf(sub)
    rows.append({'liczba_sezonów': k, 'zakres': f'{chosen[0]}-{chosen[-1]}',
                 'podia': int(sub.podium.sum()), 'ROC_AUC': auc, 'PR_AUC': ap})
wynik_clf = pd.DataFrame(rows)
wynik_clf.round(3)

dostępne sezony: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)] | wierszy: 2956 | podiów: 444


,liczba_sezonów,zakres,podia,ROC_AUC,PR_AUC
0,1,2024-2024,72,0.929,0.722
1,2,2023-2024,138,0.921,0.703
2,3,2022-2024,204,0.927,0.696
3,4,2021-2024,270,0.923,0.679
4,5,2020-2024,321,0.921,0.679
5,6,2019-2024,384,0.923,0.687
6,7,2018-2024,444,0.924,0.681


In [13]:
fig = px.line(wynik_clf, x='liczba_sezonów', y=['ROC_AUC', 'PR_AUC'], markers=True,
              title='Jakość klasyfikacji a liczba sezonów',
              labels={'liczba_sezonów': 'liczba sezonów', 'value': 'wartość metryki',
                      'variable': 'metryka'})
fig.update_yaxes(range=[0, 1])
fig.show()

Mimo że liczba podiów rośnie wraz z sezonami, jakość praktycznie stoi w miejscu - `ROC-AUC` trzyma się okolic `0.92`, a `PR-AUC` `~0.70`. Co ciekawe, podobnie jak w regresji, **dodawanie danych nie podnosi wyniku**. Przyczyna jest tu jednak inna, co najlepiej widać, gdy porównamy ery technicznie (przed i po zmianie regulaminu w 2022) oraz sprawdzimy rolę zespołu:

In [14]:
print('era vs era vs zmieszane:')
for nazwa, df in [('era >=2022', dr_all[dr_all.Season >= 2022]),
                  ('era <2022 ', dr_all[dr_all.Season < 2022]),
                  ('zmieszane ', dr_all)]:
    auc, ap = ocena_clf(df)
    print(f'  {nazwa}  podiów={int(df.podium.sum()):>3}  ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}')

print('\nrola cechy Team:')
for nazwa, ut in [('z Team  ', True), ('bez Team', False)]:
    auc, ap = ocena_clf(dr_all, use_team=ut)
    print(f'  {nazwa}  ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}')

era vs era vs zmieszane:
  era >=2022  podiów=204  ROC-AUC=0.927  PR-AUC=0.696
  era <2022   podiów=240  ROC-AUC=0.921  PR-AUC=0.699
  zmieszane   podiów=444  ROC-AUC=0.924  PR-AUC=0.681

rola cechy Team:


  z Team    ROC-AUC=0.924  PR-AUC=0.681
  bez Team  ROC-AUC=0.916  PR-AUC=0.661


Tu pojawia się **kluczowa różnica względem regresji**. W regresji mieszanie er wyraźnie szkodziło, bo cecha `Team` była drugą najważniejszą, a między erami niespójną ("Mercedes 2019" to inne tempo niż "Mercedes 2024"). W klasyfikacji jest inaczej:

- mieszanie er **nie szkodzi** - wynik na wszystkich sezonach jest taki sam jak na każdej erze osobno,
- usunięcie `Team` prawie nic nie zmienia - model opiera się głównie na `GridPosition` i `quali_gap_s`.

A te dwie cechy są **ponadczasowe**: reguła "kto startuje z przodu, ma szansę na podium" działa tak samo w 2018 i w 2024, niezależnie od regulaminu. Dlatego klasyfikacja jest odporna na zmianę ery, podczas gdy regresja na nią cierpiała.

Wniosek jest taki sam co do formy ("więcej sezonów nie pomaga"), ale inny co do przyczyny. W regresji sufit brał się z **braku cech** opisujących tempo. Tutaj `ROC-AUC ~0.92` to już bardzo wysoki poziom, a granicę wyznacza **nieprzewidywalność samej F1**: deszcz, samochód bezpieczeństwa, awarie czy odważne strategie - rzeczy, których z pozycji startowej przewidzieć się nie da (wrócimy do nich w notebooku 05).

In [15]:
out = dr.copy()
out['proba_logit'] = proba_logit
out['pred_logit'] = pred_logit
out.to_parquet('../data/clf_predictions.parquet', index=False)
bench.to_parquet('../data/clf_benchmark.parquet')
print('zapisano clf_predictions.parquet', out.shape, 'oraz clf_benchmark.parquet')

zapisano clf_predictions.parquet (918, 14) oraz clf_benchmark.parquet


## Podsumowanie

- **Sama trafność myli** przy niezbalansowanych klasach - model "zawsze nie" osiąga `~85%`, nie przewidując niczego.
- **Wyciek danych** (cecha `Points`) daje wynik pozornie idealny, lecz bezużyteczny w praktyce - dlatego trzymamy się wyłącznie cech znanych przed startem.
- Przy tej ilości danych **regresja logistyczna** wygrywa z modelami drzewiastymi jednocześnie pod względem jakości, czasu i interpretowalności.
- Najsilniejszym sygnałem okazuje się pozycja startowa wraz z różnicą czasu w kwalifikacjach - co zgadza się z intuicją kibica F1.